### Messages
Messages are the fundamental unit of context in LangChain. They represent both the input and output of models, carrying both the content and metadata needed to represent the state of conversation when interacting with LLM.<br/>
Basically they are objects or data structures that contain:
1. **Role** => Identified the message type(e.g system, user)
2. **Content** -> Represents actual content of the message (text, images etc.)
3. **Metadata** -> optional field, shich as response information, messageIDs,  and token usage.

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [3]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [6]:
os.environ["GROQ_API_KEY"] = os.getenv("API_KEY_GROQ")

In [10]:
model = init_chat_model("groq:qwen/qwen3-32b")
if model and model.profile:
    print('Model Loaded')
else:
    print(f"Error loading model. Check model name")

Model Loaded


#### Text Prompts
Text Prompts are strings. They are ideal for straightforward generation tasks where you don't need to retrain conversation history.
Use When:
1. You have a single standalone request
2. You don't need conversation history
3. You want minimal code complexity


#### Message Prompts
We can also pass list of messages to the model by providing a list of message objects.
**Message Types**
* **System Message** - Tells the model how to behave and provide context for interactions.
Represents an initial set of instructions that primes the model's behavior. Use system message to set the tone, define the model's role and estiablish guidelines for responses.
* **Human Message** - Represents user input and interactions with the model.
Represents user input and interactions. They contain text, images, audio, files and any other amount of multimodal content.
* **AI Message** - Responses generated by the model, including text content, tool calls and metadata.
This represents the output of a model invocation. They can include multimodal data, tool calls and provider-specific metadata.
* **Tool Message** - Represents the outputs of the tool calls
For models that support tool calling. Tool messages are used to pass the result of a single tool execution back to the model.

In [ ]:
### Text Prompt
model.invoke("What is lunar month?")

AIMessage(content='<think>\nOkay, so I need to figure out what a lunar month is. Let me start by recalling what I know about months. There are different types of months in astronomy, like the lunar month, the sidereal month, and the synodic month. Wait, maybe the user is asking specifically about the lunar month. I think the lunar month is related to the moon\'s phases.\n\nFirst, I should confirm the definition. A lunar month is the period it takes for the Moon to complete one orbit around the Earth relative to the Sun, which is why we see the phases of the Moon. But I need to be precise here. The phases cycle from new moon to new moon, which I believe is called the synodic month. Is that the same as the lunar month? Maybe I should check that.\n\nWait, the term "lunar month" might be another name for the synodic month. The synodic month is approximately 29.5 days long. But I should also consider other types of months. For example, the sidereal month is the time it takes the Moon to orb

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# Basic Conversation
messages = [
    SystemMessage("You are a physicist."),
    HumanMessage("Explain me 1-2 sentence different concepts in Physics related to Gravity."),
]
### We can always expand or provide more information within these messages to get more relevant LLM response.

In [13]:
response = model.invoke(messages)
response.content

"<think>\nOkay, the user wants me to explain different concepts in physics related to gravity in 1-2 sentences each. Let me start by recalling the main theories and concepts about gravity. \n\nFirst, Newton's law of universal gravitation. That's the basic one, right? It states that every mass attracts every other mass with a force proportional to their masses and inversely proportional to the square of the distance between them. I should mention the formula F = G(m1m2)/r².\n\nThen Einstein's general theory of relativity. This one is more about spacetime curvature caused by mass and energy. Maybe I can say that gravity is the result of objects moving along geodesics in this curved spacetime. Also, it predicts phenomena like gravitational time dilation and light bending.\n\nDark matter and dark energy are related to gravity in cosmology. Dark matter provides the gravitational pull needed to explain galaxy rotation curves, while dark energy is associated with the accelerated expansion of 

In [14]:
### Using Metadata
human_message = HumanMessage(content="Explain me how Physics fit into real life experience?", name="Newton", id="msg_001")
messages = [
    SystemMessage("You are a physicist and expert in explaining concepts with simple language."),
    human_message
]
response = model.invoke(messages)
response.content

'<think>\nOkay, the user is asking how physics fits into real life. I need to make sure I cover different aspects of everyday experiences where physics principles are at play. Let me start by breaking down the main areas: motion, energy, electricity, waves, and maybe some more abstract concepts like relativity or quantum for daily tech.\n\nFirst, motion is everywhere. Things moving around us—cars, people walking, even things like a ball rolling. Newton\'s laws come into play here. Maybe use an example like a car crash to explain inertia. Then energy conservation, like when you ride a bike, converting energy into motion.\n\nElectricity and magnetism are huge. Devices like phones, lights, and appliances all rely on these principles. Maybe explain how a phone charger works using electromagnetism. Waves are another big one—sound, light, radio waves. For example, how a microwave uses microwaves to heat food.\n\nThermodynamics in daily life: why coffee cools down, refrigeration. Maybe mentio

In [16]:
### Creating your own AI Message
ai_welcome = AIMessage("Thank you for your question. I am happy to help.")
messages = [
    SystemMessage("You are helpful assistant."),
    HumanMessage("Can you help me"),
    ai_welcome,
    HumanMessage("What is 2+2")
]
response = model.invoke(messages)
response.content

'<think>\nOkay, the user asked, "What is 2+2?" That\'s a basic arithmetic question. Let me make sure I understand what they\'re asking. They might just want the answer, but maybe they\'re testing my knowledge or looking for a deeper explanation.\n\nFirst, the straightforward answer is 4. But why does that happen? Let me break it down. In Peano arithmetic, numbers are defined with axioms. The number 2 can be represented as the successor of 1, which is the successor of 0. Addition is defined recursively. So 2 + 2 would be 2 + S(1), which becomes S(2 + 1), and so on until it simplifies to 4. \n\nBut maybe the user isn\'t looking for a formal proof. They might just need confirmation that the answer is indeed 4. I should check if there\'s any context I\'m missing. The user\'s previous message was "Can you help me," and now they\'re asking a math question. They might be new to arithmetic or just verifying something. \n\nI should present the answer clearly. Start with the answer, then offer a

In [17]:
response.usage_metadata

{'input_tokens': 50, 'output_tokens': 376, 'total_tokens': 426}

In [ ]:
### Tool Message
from langchain.messages import ToolMessage
ai_message = AIMessage(
    content=[],
    tool_call=[{
        "name": "get_weather",
        "args": {"location": "San Francisci"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72F" # hardcoded, but this can come from our get_weather message
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"
)

# Continue the conversation
messages = [
    HumanMessage("What's the weather in Florida?"),
    ai_message, # Model's tool call
    tool_message, # Tool execution result
]
response = model.invoke(messages)

In [19]:
response.content

'<think>\nOkay, the user asked for the weather in Florida, and I got a response of "Sunny, 72F". First, I need to present this information clearly. Let me check if there\'s any additional data needed. The user might want to know about other cities in Florida or if there are any weather alerts. But the response only mentions a general sunny and 72°F. Maybe the temperature is for a specific area? Florida has different regions with varying climates, so it\'s possible the temp is for a major city like Miami or Tampa. Should I mention that it\'s an average or a typical temperature? Also, 72°F is a mild temperature, so maybe note that it\'s comfortable. The user might be planning activities, so suggesting outdoor activities could be helpful. Also, check if there\'s any chance of rain or UV index since it\'s sunny. Maybe add a note about sunscreen. I should keep the response concise but informative. Let me structure it with the main facts first, then some additional tips.\n</think>\n\nThe cur